In [8]:
import bacdive
import json
from glob import glob
from bacdive_config import BACDIVE_EMAIL, BACDIVE_PASSWORD

client = bacdive.BacdiveClient(BACDIVE_EMAIL, BACDIVE_PASSWORD)

-- Authentication successful --


In [5]:
# Set search type: exact match on genome accession
client.setSearchType('exact')

# Genome accession numbers to search
query = {"genome": ["GCA_003332855", "GCA_024623325", "GCA_017377855"]}

# Run the search
count = client.search(**query)
print(f"🔍 {count} strains found.")

# Define the filters you want to retrieve
filters = [
    'keywords',
    'culture collection no.',
]

# Get and print the results
result = client.retrieve(filters)
parsed_result = {k: v for x in result for k, v in x.items()}

# Pretty print the result
print(json.dumps(parsed_result, indent=2))

🔍 3 strains found.
{
  "24718": [
    {
      "keywords": [
        "genome sequence",
        "16S sequence",
        "Bacteria",
        "aerobe",
        "mesophilic",
        "Gram-negative",
        "motile",
        "rod-shaped"
      ]
    },
    {
      "culture collection no.": "DSM 28897, KCTC 12899, NBRC 101209"
    }
  ],
  "23277": [
    {
      "keywords": [
        "genome sequence",
        "16S sequence",
        "Bacteria",
        "aerobe",
        "mesophilic",
        "Gram-negative",
        "rod-shaped"
      ]
    },
    {
      "culture collection no.": "DSM 18094, KCTC 12630, LMG 23739, Gsoil 634"
    }
  ],
  "18089": [
    {
      "keywords": [
        "genome sequence",
        "16S sequence",
        "Bacteria",
        "anaerobe",
        "mesophilic"
      ]
    },
    {
      "culture collection no.": "DSM 23669, ATCC BAA-2170"
    }
  ]
}


# BacDive Database Comparison Analysis

## Methodology and Data Extraction Documentation

### BacDive Version and Access Information
- **BacDive API Version**: Accessed via bacdive-python client (v1.0.3)
- **Database Access Date**: [SPECIFY_DATE_OF_EXTRACTION]
- **BacDive Database Version**: [TO_BE_DETERMINED_FROM_API]
- **Total BacDive Entries Available**: [TO_BE_QUERIED]

### Data Extraction Process
This analysis compares bacterial phenotype predictions from our LinkBERT pipeline with 
reference data from the BacDive database. **Important**: This is structured as a 
competitive comparison rather than ground truth validation, as BacDive entries 
vary in curation quality.

### Comparison Scope and Limitations
- **Entity Focus**: Primary comparison on STRAIN entities and associated phenotypes
- **Matching Strategy**: [TO_BE_DOCUMENTED] - exact vs fuzzy string matching
- **Coverage**: Analysis limited to strains with genome assembly identifiers
- **BacDive Entry Quality**: Mix of manually curated and automated entries

In [9]:
files = glob("../../assemblies_3103/*/*")

In [ ]:
# do not run
strain_to_genomes = {}

for file_path in tqdm(files):
    parts = file_path.split('/')
    strain_id = parts[-2]
    genome_id = parts[-1].replace("GCF", "GCA")

    query = {"genome": [genome_id]}
    count = client.search(**query)
    
    if count > 0:
        filters = ['keywords', 'culture collection no.']
        result = client.retrieve()
        parsed_result = {k: v for x in result for k, v in x.items()}
        
        if strain_id not in strain_to_genomes:
            strain_to_genomes[strain_id] = []

        strain_to_genomes[strain_id].append({
            "genome_id": genome_id,
            "link": f"https://www.ncbi.nlm.nih.gov/assembly/{genome_id}",
            "metadata": parsed_result
        })


In [ ]:
with open('strain_to_genomes_complete.json', 'w') as f:
    json.dump(strain_to_genomes, f, indent=2)

In [13]:
with open('strain_to_genomes.json', 'r') as f:
    strain_to_genomes = json.load(f)

In [14]:
with open('strain_to_genomes_full.json', 'r') as f:
    strain_to_genomes_full = json.load(f)

In [15]:
# First, let's get comprehensive BacDive metadata and version information
print("=== BacDive Database Information ===")
try:
    # Get BacDive version information if available
    print(f"BacDive Python Client Version: {bacdive.__version__}")
except:
    print("BacDive client version not available")

# Document the extraction date
from datetime import datetime
extraction_date = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print(f"Data extraction performed on: {extraction_date}")

# Get information about total available entries
print("\\n=== Database Statistics ===")
print(f"Analyzing {len(files)} genome assembly files")
print(f"Loaded {len(strain_to_genomes)} strain entries from previous BacDive queries")

from glob import glob
from tqdm import tqdm
from collections import defaultdict
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

=== BacDive Database Information ===
BacDive client version not available
Data extraction performed on: 2025-09-26 12:05:08
\n=== Database Statistics ===
Analyzing 41247 genome assembly files
Loaded 6872 strain entries from previous BacDive queries


In [17]:
strains_to_filter = list(strain_to_genomes.keys())

In [18]:
network = pd.read_csv("../../preds3103/network.tsv",sep="\t")

In [19]:
filtered_network = network[(network["source"].isin(strains_to_filter))|(network["target"].isin(strains_to_filter))]

In [20]:
filtered_network[filtered_network.target.str.contains("rod ")]

,source,target,rel,source_ner,target_ner
114501,t__128265,rod shaped,PRESENTS,STRAIN,PHENOTYPE
115425,t__91554,rod shaped,PRESENTS,STRAIN,PHENOTYPE
115446,t__56900,rod shaped,PRESENTS,STRAIN,PHENOTYPE
115518,t__68697,rod like,PRESENTS,STRAIN,PHENOTYPE
115694,t__30093,rod shaped,PRESENTS,STRAIN,PHENOTYPE
...,...,...,...,...,...
179052,t__118209,rod shaped,PRESENTS,STRAIN,PHENOTYPE
179209,t__30818,extracellular rod shaped,PRESENTS,STRAIN,PHENOTYPE
179779,t__78579,rod shaped,PRESENTS,STRAIN,PHENOTYPE
179922,t__154579,pleomorphic rod shaped,PRESENTS,STRAIN,PHENOTYPE


In [21]:
# Add comprehensive statistics about entity term collections
print("=== Entity Term Collection Statistics ===")

# Analyze our prediction network
print(f"\\nOur LinkBERT Network Statistics:")
print(f"Total predictions in network: {len(filtered_network)}")
print(f"Unique source entities (strains): {filtered_network['source'].nunique()}")
print(f"Unique target entities (phenotypes): {filtered_network['target'].nunique()}")
print(f"Unique relation types: {filtered_network['rel'].nunique()}")

# Analyze BacDive collection
print(f"\\nBacDive Collection Statistics:")
print(f"Total strains in BacDive subset: {len(strain_to_genomes)}")

# Count unique phenotype terms in BacDive
bacdive_phenotypes = set()
for strain_id, entries in strain_to_genomes.items():
    for entry in entries:
        metadata = entry.get('metadata', {})
        for meta_id, items in metadata.items():
            for item in items:
                if 'keywords' in item:
                    bacdive_phenotypes.update(item['keywords'])

print(f"Unique phenotype keywords in BacDive: {len(bacdive_phenotypes)}")

# Analyze overlap between collections
our_phenotypes = set(filtered_network['target'].unique())
overlap = our_phenotypes.intersection(bacdive_phenotypes)
print(f"\\nCollection Overlap:")
print(f"Phenotype terms in common: {len(overlap)}")
print(f"LinkBERT-only phenotypes: {len(our_phenotypes - bacdive_phenotypes)}")
print(f"BacDive-only phenotypes: {len(bacdive_phenotypes - our_phenotypes)}")

print(f"\\nEntity Matching Strategy Documentation:")
print(f"- Matching Type: Exact string matching (case-insensitive)")
print(f"- Conflicting Keywords: Explicitly defined opposing terms")
print(f"- Strain Matching: Based on genome assembly accession numbers")
print(f"- Quality Control: Manual review of keyword mappings")

len(strain_to_genomes)

=== Entity Term Collection Statistics ===
\nOur LinkBERT Network Statistics:
Total predictions in network: 109011
Unique source entities (strains): 11893
Unique target entities (phenotypes): 42801
Unique relation types: 11
\nBacDive Collection Statistics:
Total strains in BacDive subset: 6872
Unique phenotype keywords in BacDive: 72
\nCollection Overlap:
Phenotype terms in common: 37
LinkBERT-only phenotypes: 42764
BacDive-only phenotypes: 35
\nEntity Matching Strategy Documentation:
- Matching Type: Exact string matching (case-insensitive)
- Conflicting Keywords: Explicitly defined opposing terms
- Strain Matching: Based on genome assembly accession numbers
- Quality Control: Manual review of keyword mappings


6872

In [22]:
def evaluate_phenotype_match_enhanced(
    predictions_df,
    reference_db,
    prediction_keywords,
    reference_keyword,
    conflicting_keywords=None,
    matching_strategy="exact",
    include_provenance=False
):
    """
    Enhanced evaluation with better documentation and provenance tracking.
    
    This function implements a competitive comparison approach rather than 
    ground truth validation, recognizing that BacDive entries vary in quality.
    
    Args:
        predictions_df (pd.DataFrame): LinkBERT predictions with 'source', 'target', 'rel' columns
        reference_db (dict): BacDive strain metadata 
        prediction_keywords (list[str]): Phenotype terms to evaluate from predictions
        reference_keyword (str): Corresponding BacDive keyword
        conflicting_keywords (list[str], optional): Terms indicating false positives
        matching_strategy (str): "exact" or "fuzzy" matching approach
        include_provenance (bool): Whether to track BacDive entry sources
        
    Returns:
        tuple: (metrics_df, detailed_results_df)
    """
    
    prediction_keywords = set(prediction_keywords)
    conflicting_keywords = set(conflicting_keywords or [])
    
    # Extract BacDive keywords with optional provenance
    def extract_keywords_with_metadata(db):
        strain_keywords = defaultdict(set)
        strain_provenance = defaultdict(list) if include_provenance else None
        
        for strain, entries in db.items():
            for entry in entries:
                metadata = entry.get('metadata', {})
                # Store genome accession for provenance
                genome_id = entry.get('genome_id', 'unknown')
                
                for meta_id, items in metadata.items():
                    for item in items:
                        if 'keywords' in item:
                            strain_keywords[strain].update(item['keywords'])
                            if include_provenance:
                                strain_provenance[strain].append({
                                    'genome_id': genome_id,
                                    'bacdive_id': meta_id,
                                    'keywords': item['keywords']
                                })
        
        return strain_keywords, strain_provenance
    
    strain_to_keywords, strain_provenance = extract_keywords_with_metadata(reference_db)
    
    # Get all strains from predictions
    all_strains = predictions_df['source'].unique()
    
    results = []
    detailed_results = []
    
    for strain in all_strains:
        # BacDive reference
        keywords = strain_to_keywords.get(strain, set())
        has_reference = reference_keyword in keywords
        has_conflict = bool(keywords & conflicting_keywords)
        
        # LinkBERT predictions
        predicted_rows = predictions_df[predictions_df['source'] == strain]
        
        if matching_strategy == "exact":
            predicted = any(target in prediction_keywords for target in predicted_rows['target'])
        else:
            # Fuzzy matching implementation would go here
            predicted = any(any(pred_kw.lower() in target.lower() 
                              for pred_kw in prediction_keywords) 
                           for target in predicted_rows['target'])
        
        # Classification logic
        if has_conflict and predicted:
            true_label, pred_label = False, True  # False positive due to conflict
            classification = "False Positive (Conflict)"
        elif has_reference:
            true_label, pred_label = True, predicted
            classification = "True Positive" if predicted else "False Negative"
        elif predicted:
            true_label, pred_label = False, True  # False positive
            classification = "False Positive"
        else:
            true_label, pred_label = False, False  # True negative
            classification = "True Negative"
        
        results.append((true_label, pred_label))
        
        # Detailed tracking
        detailed_results.append({
            'strain': strain,
            'has_bacdive_reference': has_reference,
            'has_conflict': has_conflict,
            'linkbert_predicted': predicted,
            'classification': classification,
            'bacdive_keywords': list(keywords) if keywords else [],
            'predicted_terms': list(predicted_rows['target']) if len(predicted_rows) > 0 else []
        })
    
    if not results:
        return pd.DataFrame([{
            'Prediction Keywords': ', '.join(prediction_keywords),
            'Reference Keyword': reference_keyword,
            'Matching Strategy': matching_strategy,
            'Support': 0, 'Accuracy': 0.0, 'Precision': 0.0, 
            'Recall': 0.0, 'F1 Score': 0.0, 'Total Cases': 0
        }]), pd.DataFrame()
    
    y_true, y_pred = zip(*results)
    
    metrics = {
        'Prediction Keywords': ', '.join(prediction_keywords),
        'Reference Keyword': reference_keyword,
        'Conflicting Keywords': ', '.join(conflicting_keywords) if conflicting_keywords else None,
        'Matching Strategy': matching_strategy,
        'Support (True Positives)': sum(y_true),
        'Total BacDive Strains': sum(1 for strain in all_strains if strain in strain_to_keywords),
        'Total LinkBERT Strains': len(all_strains),
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred, zero_division=0),
        'Recall': recall_score(y_true, y_pred, zero_division=0),
        'F1 Score': f1_score(y_true, y_pred, zero_division=0),
        'Total Cases': len(results)
    }
    
    return pd.DataFrame([metrics]), pd.DataFrame(detailed_results)


def analyze_bacdive_entry_quality(reference_db):
    """
    Analyze the quality and source of BacDive entries to understand curation levels.
    
    Returns:
        dict: Statistics about entry quality and sources
    """
    
    quality_stats = {
        'total_entries': 0,
        'entries_with_genome_links': 0,
        'entries_with_culture_collections': 0,
        'avg_keywords_per_entry': 0,
        'keyword_distribution': defaultdict(int)
    }
    
    all_keywords = []
    
    for strain, entries in reference_db.items():
        quality_stats['total_entries'] += len(entries)
        
        for entry in entries:
            # Check for genome links (indicator of automated curation)
            if entry.get('genome_id'):
                quality_stats['entries_with_genome_links'] += 1
            
            metadata = entry.get('metadata', {})
            for meta_id, items in metadata.items():
                for item in items:
                    # Check for culture collection info (indicator of manual curation)
                    if 'culture collection no.' in item:
                        quality_stats['entries_with_culture_collections'] += 1
                    
                    # Count keywords
                    if 'keywords' in item:
                        keywords = item['keywords']
                        all_keywords.extend(keywords)
                        for kw in keywords:
                            quality_stats['keyword_distribution'][kw] += 1
    
    quality_stats['avg_keywords_per_entry'] = len(all_keywords) / max(quality_stats['total_entries'], 1)
    
    return quality_stats

In [23]:
def evaluate_phenotype_match(
    predictions_df,
    reference_db,
    prediction_keywords,
    reference_keyword,
    conflicting_keywords=None
):
    """
    Evaluate predictions for a given phenotype with proper recall calculation.

    Args:
        predictions_df (pd.DataFrame): Must have 'source' and 'target' columns.
        reference_db (dict): Strain metadata.
        prediction_keywords (list[str]): List of phrases to detect in 'target'.
        reference_keyword (str): Keyword indicating true label in reference.
        conflicting_keywords (list[str], optional): Keywords marking explicit false positives.

    Returns:
        pd.DataFrame: Summary with precision, recall, F1, support, etc.
    """

    # Keyword prep
    prediction_keywords = set(prediction_keywords)
    conflicting_keywords = set(conflicting_keywords or [])

    # Extract reference keywords per strain
    def extract_keywords(db):
        strain_keywords = defaultdict(set)
        for strain, entries in db.items():
            for entry in entries:
                metadata = entry.get('metadata', {})
                for meta_id, items in metadata.items():
                    for item in items:
                        if 'keywords' in item:
                            strain_keywords[strain].update(item['keywords'])
        return strain_keywords

    strain_to_keywords = extract_keywords(reference_db)

    # Get all unique strain IDs from predictions
    all_strains = predictions_df['source'].unique()

    # Create binary ground truth and prediction labels per strain
    results = []
    for strain in all_strains:
        # Ground truth label
        keywords = strain_to_keywords.get(strain, set())
        has_ref = reference_keyword in keywords
        has_conflict = bool(keywords & conflicting_keywords)

        # Predicted label
        predicted_rows = predictions_df[predictions_df['source'] == strain]
        predicted = any(any(pred_kw in t for pred_kw in prediction_keywords) for t in predicted_rows['target'])

        if has_conflict and predicted:
            results.append((False, True))  # false positive due to conflict
        elif has_ref:
            results.append((True, predicted))  # match or false negative
        elif predicted:
            results.append((False, True))  # false positive
        else:
            results.append((False, False))  # true negative

    if not results:
        return pd.DataFrame([{
            'Prediction Keywords': ', '.join(prediction_keywords),
            'Reference Keyword': reference_keyword,
            'Conflicting Keywords': ', '.join(conflicting_keywords),
            'Support': 0,
            'Accuracy': 0.0,
            'Precision': 0.0,
            'Recall': 0.0,
            'F1 Score': 0.0,
            'Ignored Cases': 0
        }])

    y_true, y_pred = zip(*results)

    metrics = {
        'Prediction Keywords': ', '.join(prediction_keywords),
        'Reference Keyword': reference_keyword,
        'Conflicting Keywords': ', '.join(conflicting_keywords) if conflicting_keywords else None,
        'Support': sum(y_true),
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred, zero_division=0),
        'Recall': recall_score(y_true, y_pred, zero_division=0),
        'F1 Score': f1_score(y_true, y_pred, zero_division=0),
        'Total Cases': len(results)
    }

    return pd.DataFrame([metrics])

In [35]:
filtered_network

,source,target,rel,source_ner,target_ner
0,t__69045,geothermally heated,INHABITS,STRAIN,ISOLATE
1,t__265295,shallow acidic pool,INHABITS,STRAIN,ISOLATE
28,t__89135,human,INHABITS,STRAIN,ISOLATE
34,t__68697,sediment of a,INHABITS,STRAIN,ISOLATE
35,t__68697,surface lake,INHABITS,STRAIN,ISOLATE
...,...,...,...,...,...
490214,t__12278,staphylococcus epidermidis,INHIBITS,STRAIN,SPECIES
490225,t__33654,b. subtilis,INHIBITS,STRAIN,SPECIES
490226,t__33654,s. epidermidis,INHIBITS,STRAIN,SPECIES
490227,t__33654,proteus vulgaris,INHIBITS,STRAIN,SPECIES


In [32]:
evaluate_phenotype_match(
    predictions_df=filtered_network,
    reference_db=strain_to_genomes,
    prediction_keywords=["gram negative"],
    reference_keyword="Gram-negative",
    conflicting_keywords=["Gram-positive","Gram-variable"]
)

,Prediction Keywords,Reference Keyword,Conflicting Keywords,Support,Accuracy,Precision,Recall,F1 Score,Total Cases
0,gram negative,Gram-negative,"Gram-variable, Gram-positive",3285,0.750441,0.721678,0.157078,0.258,11893


In [26]:
vertices = pd.read_csv("../../preds3103/strainselect/StrainSelect21_vertices.tab.txt",sep="\t")

/scratch/slurm_tmpdir/job_1549359/ipykernel_563531/1527655825.py:1: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  vertices = pd.read_csv("../../preds3103/strainselect/StrainSelect21_vertices.tab.txt",sep="\t")


In [27]:
filtered_network[filtered_network.target.str.contains("aerob")].value_counts("target").head(30)

target
anaerobic                         469
aerobic                           439
facultative anaerobic             119
strictly anaerobic                117
microaerobic                       62
strictly aerobic                   51
anaerobic respiration              40
strict anaerobic                   29
gifu anaerobic medium              28
obligate anaerobic                 27
obligate anaerobe                  25
anaerobiosis                       21
aerobic gram negative              17
aerobic gram positive              17
strict anaerobe                    16
obligately anaerobic               15
grown aerobically                  15
gifu anaerobic                     14
facultative anaerobes              13
aerobic conditions                 12
micro aerobic                      11
obligate aerobe                    11
anaerobic conditions               10
strictly anaerobic conditions      10
anaerobic growth                   10
aerobic cellulolytic                9
aerob

In [29]:
evaluations = [
    {
        "prediction_keywords": ["human pathogenic"],
        "reference_keyword": "human pathogen",
        "conflicting_keywords": ["plant pathogen"],
    },
    {
        "prediction_keywords": ["plant pathogenic"],
        "reference_keyword": "plant pathogen",
        "conflicting_keywords": ["human pathogen"],
    },
    {
        "prediction_keywords": ["gram positive","g positive","gram stain positive","gram positive cocci","gram positive coccus","filamentous gram positive","gram positive, nonmotile","aerobic gram positive","both gram positive",
                                "gram positive, endospore forming","hemolytic gram positive","gram positive (g+","gram positive bacteria","gram positive actinobacterium","gram (+","gram +"],
        "reference_keyword": "Gram-positive",
        "conflicting_keywords": ["Gram-negative","Gram-variable"],
    },
    {
        "prediction_keywords": ["gram negative","g negative","gram stain negative","gram negative cocci","aerobic gram negative","gram staining negative"],
        "reference_keyword": "Gram-negative",
        "conflicting_keywords": ["Gram-positive","Gram-variable"],
    },
    {
        "prediction_keywords": ["motile","motile gram negative","motile rod shaped","motile rods"],
        "reference_keyword": "motile",
        "conflicting_keywords": None,
    },
    {
        "prediction_keywords": ["spore","spore forming"],
        "reference_keyword": "spore-forming",
        "conflicting_keywords": None,
    },
    {
        "prediction_keywords": ["rod shaped","rods","short rod shaped", "rod like"],
        "reference_keyword": "rod-shaped",
        "conflicting_keywords": ["coccus shaped","ovoid shaped","sphere shaped","vibrio shaped"],
    },
    {
        "prediction_keywords": ["mesophilic","mesophilic anaerobic"],
        "reference_keyword": "mesophilic",
        "conflicting_keywords": ["psychrophilic","thermophilic"],
    },
    {
        "prediction_keywords": ["aerobic"],
        "reference_keyword": "aerobe",
        "conflicting_keywords": ["anaerobe"],
    },
    {
        "prediction_keywords": ["anaerobic"],
        "reference_keyword": "anaerobe",
        "conflicting_keywords": ["aerobe"],
    },
    {
        "prediction_keywords": ["facultative anaerobic","facultative anaerobes","facultative anaerobe"],
        "reference_keyword": "facultative anaerobe",
        "conflicting_keywords": ["obligate anaerobe","obligate aerobe"],
    },
    {
        "prediction_keywords": ["obligate anaerobic","obligate anaerobes","obligate anaerobe","strict anaerobe","strictly anaerobic"],
        "reference_keyword": "obligate anaerobe",
        "conflicting_keywords": ["facultative anaerobe","facultative aerobe"],
    },
]

# Collect results
results = []

for eval in evaluations:
    result = evaluate_phenotype_match(
        predictions_df=filtered_network,
        reference_db=strain_to_genomes,
        prediction_keywords=eval["prediction_keywords"],
        reference_keyword=eval["reference_keyword"],
        conflicting_keywords=eval["conflicting_keywords"],
    )
    
    # Add metadata to the result using assign
    result = result.assign(
        prediction_keywords=[eval["prediction_keywords"]] * len(result),
        reference_keyword=[eval["reference_keyword"]] * len(result),
        conflicting_keywords=[eval["conflicting_keywords"]] * len(result),
    )
    
    results.append(result)

# Combine all results into a single DataFrame
results_df = pd.concat(results, ignore_index=True)

In [30]:
results_df

,Prediction Keywords,Reference Keyword,Conflicting Keywords,Support,Accuracy,Precision,Recall,F1 Score,Total Cases,prediction_keywords,reference_keyword,conflicting_keywords
0,human pathogenic,human pathogen,plant pathogen,250,0.978811,0.454545,0.040000,0.073529,11893,[human pathogenic],human pathogen,[plant pathogen]
1,plant pathogenic,plant pathogen,human pathogen,108,0.990919,0.500000,0.046296,0.084746,11893,[plant pathogenic],plant pathogen,[human pathogen]
2,"gram positive, nonmotile, gram positive (g+, g...",Gram-positive,"Gram-variable, Gram-negative",1560,0.865635,0.472543,0.209615,0.290409,11893,"[gram positive, g positive, gram stain positiv...",Gram-positive,"[Gram-negative, Gram-variable]"
3,"gram negative, gram negative cocci, gram stain...",Gram-negative,"Gram-variable, Gram-positive",3285,0.751787,0.722296,0.164688,0.268220,11893,"[gram negative, g negative, gram stain negativ...",Gram-negative,"[Gram-positive, Gram-variable]"
4,"motile rods, motile, motile rod shaped, motile...",motile,None,1388,0.868158,0.225610,0.053314,0.086247,11893,"[motile, motile gram negative, motile rod shap...",motile,None
5,"spore forming, spore",spore-forming,None,1105,0.907172,0.501160,0.195475,0.281250,11893,"[spore, spore forming]",spore-forming,None
6,"rod shaped, rods, short rod shaped, rod like",rod-shaped,"sphere shaped, ovoid shaped, vibrio shaped, co...",2561,0.783066,0.472622,0.064037,0.112792,11893,"[rod shaped, rods, short rod shaped, rod like]",rod-shaped,"[coccus shaped, ovoid shaped, sphere shaped, v..."
7,"mesophilic anaerobic, mesophilic",mesophilic,"psychrophilic, thermophilic",5142,0.573110,0.679558,0.023921,0.046215,11893,"[mesophilic, mesophilic anaerobic]",mesophilic,"[psychrophilic, thermophilic]"
8,aerobic,aerobe,anaerobe,1937,0.768351,0.187786,0.127001,0.151524,11893,[aerobic],aerobe,[anaerobe]
9,anaerobic,anaerobe,aerobe,1218,0.909274,0.586550,0.386700,0.466106,11893,[anaerobic],anaerobe,[aerobe]


In [36]:
import re
from collections import defaultdict
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score


def evaluate_phenotype_match_regex(
    predictions_df,
    reference_db,
    prediction_keywords,
    reference_keyword,
    conflicting_keywords=None
):
    """
    Evaluate predictions for a given phenotype using case-insensitive substring/regex matching.

    Args:
        predictions_df (pd.DataFrame): Must have 'source' and 'target' columns.
        reference_db (dict): Strain metadata.
        prediction_keywords (list[str]): List of regex/substring patterns to detect in 'target'.
        reference_keyword (str): Keyword indicating true label in reference.
        conflicting_keywords (list[str], optional): Keywords marking explicit false positives.

    Returns:
        pd.DataFrame: Summary with precision, recall, F1, support, etc.
    """

    conflicting_keywords = conflicting_keywords or []

    # Extract reference keywords per strain
    def extract_keywords(db):
        strain_keywords = defaultdict(set)
        for strain, entries in db.items():
            for entry in entries:
                metadata = entry.get('metadata', {})
                for meta_id, items in metadata.items():
                    for item in items:
                        if 'keywords' in item:
                            strain_keywords[strain].update(item['keywords'])
        return strain_keywords

    strain_to_keywords = extract_keywords(reference_db)

    all_strains = predictions_df['source'].unique()
    results = []

    for strain in all_strains:
        keywords = strain_to_keywords.get(strain, set())
        has_ref = reference_keyword in keywords
        has_conflict = any(
            re.search(conf_kw, " ".join(keywords), flags=re.IGNORECASE)
            for conf_kw in conflicting_keywords
        )

        predicted_rows = predictions_df[predictions_df['source'] == strain]
        predicted = any(
            re.search(pred_kw, t, flags=re.IGNORECASE)
            for pred_kw in prediction_keywords
            for t in predicted_rows['target']
        )

        if has_conflict and predicted:
            results.append((False, True))  # false positive due to conflict
        elif has_ref:
            results.append((True, predicted))  # match or false negative
        elif predicted:
            results.append((False, True))  # false positive
        else:
            results.append((False, False))  # true negative

    if not results:
        return pd.DataFrame([{
            'Prediction Keywords': ', '.join(prediction_keywords),
            'Reference Keyword': reference_keyword,
            'Conflicting Keywords': ', '.join(conflicting_keywords),
            'Support': 0,
            'Accuracy': 0.0,
            'Precision': 0.0,
            'Recall': 0.0,
            'F1 Score': 0.0,
            'Total Cases': 0
        }])

    y_true, y_pred = zip(*results)

    metrics = {
        'Prediction Keywords': ', '.join(prediction_keywords),
        'Reference Keyword': reference_keyword,
        'Conflicting Keywords': ', '.join(conflicting_keywords) if conflicting_keywords else None,
        'Support': sum(y_true),
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred, zero_division=0),
        'Recall': recall_score(y_true, y_pred, zero_division=0),
        'F1 Score': f1_score(y_true, y_pred, zero_division=0),
        'Total Cases': len(results)
    }

    return pd.DataFrame([metrics])


# --- Evaluation definitions ---
evaluations = [
    {
        "prediction_keywords": ["human pathog"],
        "reference_keyword": "human pathogen",
        "conflicting_keywords": ["plant pathog"],
    },
    {
        "prediction_keywords": ["plant pathog"],
        "reference_keyword": "plant pathogen",
        "conflicting_keywords": ["human pathog"],
    },
    {
        "prediction_keywords": ["gram positive"],
        "reference_keyword": "Gram-positive",
        "conflicting_keywords": ["gram negative", "gram variable"],
    },
    {
        "prediction_keywords": ["gram negative"],
        "reference_keyword": "Gram-negative",
        "conflicting_keywords": ["gram positive", "gram variable"],
    },
    {
        "prediction_keywords": ["motile"],
        "reference_keyword": "motile",
        "conflicting_keywords": None,
    },
    {
        "prediction_keywords": ["spore"],
        "reference_keyword": "spore-forming",
        "conflicting_keywords": None,
    },
    {
        "prediction_keywords": ["rod"],
        "reference_keyword": "rod-shaped",
        "conflicting_keywords": ["coccus", "ovoid", "sphere", "vibrio"],
    },
    {
        "prediction_keywords": ["mesophilic"],
        "reference_keyword": "mesophilic",
        "conflicting_keywords": ["psychrophilic", "thermophilic"],
    },
    {
        "prediction_keywords": ["aerobic"],
        "reference_keyword": "aerobe",
        "conflicting_keywords": ["anaerobe"],
    },
    {
        "prediction_keywords": ["anaerobic"],
        "reference_keyword": "anaerobe",
        "conflicting_keywords": ["aerobe"],
    },
    {
        "prediction_keywords": ["facultative anaerob"],
        "reference_keyword": "facultative anaerobe",
        "conflicting_keywords": ["obligate anaerob", "obligate aerob"],
    },
    {
        "prediction_keywords": ["obligate anaerob", "strict anaerob"],
        "reference_keyword": "obligate anaerobe",
        "conflicting_keywords": ["facultative anaerob", "facultative aerob"],
    },
]


# --- Run evaluations ---
results = []

for eval in evaluations:
    result = evaluate_phenotype_match_regex(
        predictions_df=filtered_network,
        reference_db=strain_to_genomes,
        prediction_keywords=eval["prediction_keywords"],
        reference_keyword=eval["reference_keyword"],
        conflicting_keywords=eval["conflicting_keywords"],
    )

    results.append(result)

# Combine all results into a single DataFrame
phenotype_matches_df = pd.concat(results, ignore_index=True)

In [37]:
phenotype_matches_df

,Prediction Keywords,Reference Keyword,Conflicting Keywords,Support,Accuracy,Precision,Recall,F1 Score,Total Cases
0,human pathog,human pathogen,plant pathog,250,0.978727,0.434783,0.040000,0.073260,11893
1,plant pathog,plant pathogen,human pathog,108,0.990415,0.312500,0.046296,0.080645,11893
2,gram positive,Gram-positive,"gram negative, gram variable",1560,0.865299,0.469027,0.203846,0.284182,11893
3,gram negative,Gram-negative,"gram positive, gram variable",3285,0.750441,0.721678,0.157078,0.258000,11893
4,motile,motile,None,1388,0.868158,0.225610,0.053314,0.086247,11893
5,spore,spore-forming,None,1105,0.907172,0.501160,0.195475,0.281250,11893
6,rod,rod-shaped,"coccus, ovoid, sphere, vibrio",2560,0.779114,0.450952,0.120313,0.189948,11893
7,mesophilic,mesophilic,"psychrophilic, thermophilic",5142,0.573110,0.679558,0.023921,0.046215,11893
8,aerobic,aerobe,anaerobe,1932,0.767931,0.183969,0.124741,0.148674,11893
9,anaerobic,anaerobe,aerobe,747,0.869671,0.000000,0.000000,0.000000,11893


In [38]:
# Analyze BacDive entry quality and curation levels
print("=== BacDive Entry Quality Analysis ===")
quality_stats = analyze_bacdive_entry_quality(strain_to_genomes)

print(f"Total BacDive entries analyzed: {quality_stats['total_entries']}")
print(f"Entries with genome links: {quality_stats['entries_with_genome_links']} ({quality_stats['entries_with_genome_links']/quality_stats['total_entries']*100:.1f}%)")
print(f"Entries with culture collection info: {quality_stats['entries_with_culture_collections']} ({quality_stats['entries_with_culture_collections']/quality_stats['total_entries']*100:.1f}%)")
print(f"Average keywords per entry: {quality_stats['avg_keywords_per_entry']:.1f}")

print(f"\\nMost common BacDive keywords:")
for keyword, count in sorted(quality_stats['keyword_distribution'].items(), key=lambda x: x[1], reverse=True)[:15]:
    print(f"  {keyword}: {count}")

print(f"\\n=== Relationship Type Analysis ===")
print("LinkBERT Network Relation Types:")
rel_counts = filtered_network['rel'].value_counts()
for rel, count in rel_counts.head(10).items():
    print(f"  {rel}: {count}")

print(f"\\n=== Entity Type Distribution ===")
print("LinkBERT Entity Types (source/target):")
print(f"Source NER types: {filtered_network['source_ner'].value_counts().to_dict()}")
print(f"Target NER types: {filtered_network['target_ner'].value_counts().to_dict()}")

# Compare with different entity types if available
isolate_predictions = filtered_network[filtered_network['source_ner'] == 'ISOLATE'] if 'ISOLATE' in filtered_network['source_ner'].values else pd.DataFrame()
medium_predictions = filtered_network[filtered_network['target_ner'] == 'MEDIUM'] if 'MEDIUM' in filtered_network['target_ner'].values else pd.DataFrame()

print(f"\\nISOLATE entity predictions: {len(isolate_predictions)}")
print(f"MEDIUM entity predictions: {len(medium_predictions)}")

if len(isolate_predictions) > 0:
    print("Sample ISOLATE predictions:")
    print(isolate_predictions[['source', 'target', 'rel']].head())

if len(medium_predictions) > 0:
    print("Sample MEDIUM predictions:")
    print(medium_predictions[['source', 'target', 'rel']].head())

=== BacDive Entry Quality Analysis ===
Total BacDive entries analyzed: 9352
Entries with genome links: 9352 (100.0%)
Entries with culture collection info: 9347 (99.9%)
Average keywords per entry: 6.3
\nMost common BacDive keywords:
  genome sequence: 9352
  Bacteria: 9006
  16S sequence: 8421
  mesophilic: 7343
  Gram-negative: 4421
  rod-shaped: 3784
  aerobe: 2632
  Gram-positive: 2363
  motile: 1948
  anaerobe: 1713
  spore-forming: 1482
  facultative anaerobe: 712
  obligate aerobe: 673
  microaerophile: 656
  coccus-shaped: 616
\n=== Relationship Type Analysis ===
LinkBERT Network Relation Types:
  PRESENTS: 29083
  GROWS_ON: 20748
  INHIBITS: 14996
  INHABITS: 12596
  PRODUCES: 12058
  RESISTS: 5766
  PROMOTES: 5064
  DEGRADES: 3726
  INFECTS: 3569
  ASSOCIATED_WITH: 873
\n=== Entity Type Distribution ===
LinkBERT Entity Types (source/target):
Source NER types: {'STRAIN': 97903, 'COMPOUND': 11108}
Target NER types: {'COMPOUND': 21550, 'MEDIUM': 20748, 'EFFECT': 17461, 'PHENOTYPE'